# Loading and Querying Datasets

This tutorial covers how to load datasets, inspect their structure, and query entities from them. Datasets are the primary interface for working with EDA-schema data stored in databases.

## What you'll learn

This tutorial covers:
- Initialize a dataset with a database backend
- Load and inspect dataset structure
- Query entities by flow_id and stage
- Filter and search data efficiently
- Understand dataset organization (flows → stages → entities)

## Requirements

- Completed Tutorials 1 & 2
- EDA-schema installed
- A dataset directory (or create one with `examples/01_basics/02_create_dataset.py`)

## See also

- **Python Examples**: `examples/01_basics/03_load_dataset.py`, `examples/03_datasets/`

In [24]:
# Import required modules
from eda_schema.dataset import Dataset
from eda_schema.db import ParquetDB
from pathlib import Path

print("Imports successful!")

Imports successful!


## Part 1: Initializing a Dataset

A `Dataset` is a container that manages design flows, stages, and entities. It connects to a database backend (ParquetDB, FileDB, etc.) where data is stored.

**Dataset Structure:**
```
Dataset
  └── DesignFlow (flow_id)
      └── DesignStage (stage)
          ├── NetlistEntity
          ├── Metrics (Area, Power, Timing, etc.)
          └── Other entities
```

**Note:** Update `DATASET_DIR` to point to your dataset, or use the minimal dataset in `examples/data/minimal_dataset/`.


In [25]:
# Initialize dataset
# Update this path to point to your dataset directory
DATASET_DIR = "../../dataset/test"  # Adjust path as needed

# Create database connection
db = ParquetDB(DATASET_DIR)

# Create dataset
dataset = Dataset(db)

print(f"Dataset initialized with ParquetDB")
print(f"   Dataset path: {DATASET_DIR}")

# Load standard cells (if available)
dataset.load_standard_cells()
print(f"Loaded {len(dataset.standard_cells)} standard cells")

Dataset initialized with ParquetDB
   Dataset path: ../../dataset/test
Loaded 441 standard cells


## Part 2: Inspecting Dataset Structure

Before loading data, it's useful to inspect what's available in the dataset. You can check:
- Available design flows
- Available stages for each flow
- Entity counts
- Database tables


In [14]:
# Check available design flows
flows_df = dataset.db.get_table_data('design_flows')
print("Available Design Flows:")
print(flows_df[['flow_id', 'design', 'run_status']].to_string(index=False))
print(f"\n   Total flows: {len(flows_df)}")

Available Design Flows:
   flow_id design run_status
gcd-000001    gcd   Complete

   Total flows: 1


In [16]:
# Check available stages
stages_df = dataset.db.get_table_data('design_stages')
print("Available Design Stages:")
stages_summary = stages_df.groupby('flow_id')['stage'].apply(list).to_dict()
for flow_id, stages in list(stages_summary.items())[:5]:  # Show first 5
    print(f"   {flow_id}: {stages}")
if len(stages_summary) > 5:
    print(f"   ... and {len(stages_summary) - 5} more flows")
print(f"\n   Total stages: {len(stages_df)}")

Available Design Stages:
   gcd-000001: ['floorplan', 'global_place', 'place_resized', 'detailed_place', 'cts', 'global_route', 'detailed_route', 'final']

   Total stages: 8


## Part 3: Loading Data

There are several ways to load data from a dataset:

1. **Load entire dataset**: `dataset.load()` - Loads all flows and stages
2. **Load specific flow**: `dataset.load(flow_id='...')` - Loads one flow
3. **Load specific stage**: `dataset.load(flow_id='...', stage='...')` - Loads one stage
4. **Load individual entities**: `dataset.load_netlist(...)`, `dataset.load_design_stage(...)`

**Best Practice:** Load only what you need to save memory and time.


In [18]:
# Get a flow_id to work with
flows_df = dataset.db.get_table_data('design_flows')
flow_id = flows_df.iloc[0]['flow_id']
print(f"  Using flow_id: {flow_id}")

  Using flow_id: gcd-000001


In [19]:
# Try to load this flow
dataset.load(flow_id=flow_id)
design_flow = dataset[flow_id]
print(f"Loaded design flow: {flow_id}")
print(f"   Design: {design_flow.design}")
print(f"   Stages: {list(design_flow.stages.keys())}")

Loaded design flow: gcd-000001
   Design: gcd
   Stages: ['floorplan', 'global_place', 'place_resized', 'detailed_place', 'cts', 'global_route', 'detailed_route', 'final']


### Loading a Netlist

A netlist is the graph representation of the circuit. You can load it directly using `load_netlist()`.


In [20]:
# Load a netlist
# Update stage as needed: 'floorplan', 'global_place', 'detailed_place', 'cts', 'global_route', 'detailed_route', 'final'
stage = "final"

netlist = dataset.load_netlist(flow_id=flow_id, stage=stage)
print(f"Loaded netlist for flow '{flow_id}' at stage '{stage}'")
print(f"   Cells: {netlist.no_of_cells}")
print(f"   Nets: {netlist.no_of_nets}")
print(f"   Pins: {netlist.no_of_pins}")
print(f"   Utilization: {netlist.utilization:.2%}")
print(f"   Size: {netlist.width} x {netlist.height} um")

Loaded netlist for flow 'gcd-000001' at stage 'final'
   Cells: 1006
   Nets: 413
   Pins: 1081
   Utilization: 9800.00%
   Size: 97.34 x 97.34 um


## Part 4: Querying Entities

You can query entities directly from the database without loading the entire dataset. This is efficient for extracting specific information.


In [23]:
# Query gates for a specific flow and stage
gates_df = dataset.db.get_table_data('gates', flow_id=flow_id, stage=stage)
print(f"Gates for flow '{flow_id}' at stage '{stage}':")
print(f"   Total gates: {len(gates_df)}")
print(f"\n   Sample gates:")
gates_df[['name', 'standard_cell', 'x_min', 'y_min']].head()

Gates for flow 'gcd-000001' at stage 'final':
   Total gates: 1006

   Sample gates:


,name,standard_cell,x_min,y_min
0,FILLER_0_0_0,sky130_fd_sc_hd__fill_8,5.06,5.44
1,FILLER_0_0_135,sky130_fd_sc_hd__fill_4,67.16,5.44
2,FILLER_0_0_139,sky130_fd_sc_hd__fill_2,69.00,5.44
3,FILLER_0_0_141,sky130_fd_sc_hd__fill_1,69.92,5.44
4,FILLER_0_0_145,sky130_fd_sc_hd__fill_4,71.76,5.44


### Querying Specific Entities

You can get a single entity by its primary keys (flow_id, stage, name).


In [26]:
# Get a specific gate
gates_df = dataset.db.get_table_data('gates', flow_id=flow_id, stage=stage)
gate_name = gates_df.iloc[0]['name']
gate = dataset.db.get_entity('gates', flow_id=flow_id, stage=stage, name=gate_name)
print(f"Retrieved gate: {gate.name}")
print(f"   Standard cell: {gate.standard_cell}")
print(f"   Position: ({gate.x_min}, {gate.y_min}) to ({gate.x_max}, {gate.y_max})")

Retrieved gate: FILLER_0_0_0
   Standard cell: sky130_fd_sc_hd__fill_8
   Position: (5.06, 5.44) to (8.74, 8.16)


## Summary

Summary:

**Dataset Initialization**: How to create a Dataset with a database backend
**Structure Inspection**: How to explore available flows and stages
**Loading Data**: Different methods to load data (full, flow, stage, entity)
**Querying Entities**: How to query specific entities from the database

## Key points

1. **Load Only What You Need**: Use specific flow_id and stage filters to save memory
2. **Inspect First**: Check available data before loading
3. **Query vs Load**: Query for simple data extraction, load for graph operations
4. **Database Backends**: ParquetDB is recommended for most use cases

## What's next

- **Tutorial 4**: Learn graph operations and traversal
- **Examples**: Try `examples/03_datasets/` for more query patterns
- **User Guide**: Read `docs/guides/user_guide.md` for complete reference
